In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm, colors
import glob

from numba import njit, vectorize, float32

from typing import Callable, Optional, Union

import hoomd
import gsd.hoomd
import freud

import sys
import time
import pickle
import gc
import warnings
import copy
import pathlib

import os
import sys

import signac

from dataclasses import dataclass
from dataclasses_json import dataclass_json

In [2]:
from monk import nb, prep, pair, render, utils, workflow, grid

config = workflow.get_config()

In [3]:
project: signac.Project = signac.init_project(name="mlj-production", root=config['root'])
project.doc

{'ref_sim_temps': [0.41, 0.43, 0.45, 0.47, 0.49, 0.51, 0.53, 0.55, 0.58, 0.61, 0.64, 0.67, 0.7, 0.75, 0.8, 0.9, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0], 'avail_seed': 18, 'alpha_iters': 40, 'dt': 0.001, 'usteps': 1000, '_status': {}}

In [5]:
project.doc["usteps"]

1000

In [13]:
for job in project.find_jobs({"delta": 0.0, "replica": 0.0}, {"_CRYSTAL": {"$exists": False}}):
    print("Job", job.sp)

    sp = job.sp
    doc = job.doc
    
    assert isinstance(job, signac.Project.Job)

    runs = sorted(glob.glob(job.fn("runs/temp-*/traj.gsd")))[:10]

    for run in runs:

        temp = utils.extract_between(run, "temp-", "/traj.gsd")
        out_dir = f"long_runs/temp-{temp}"
        os.makedirs(job.fn(out_dir), exist_ok=True)
        done = f"{out_dir}/DONE"
        if job.isfile(done):
            print("\tTemp is done!", temp)
            continue

        print("\tTemp:", temp)

        sim = hoomd.Simulation(hoomd.device.GPU(), seed=doc["seed"])
        sim.create_state_from_gsd(filename=run)

        integrator = hoomd.md.Integrator(dt=project.doc["dt"])
        tree = hoomd.md.nlist.Tree(0.3)
        mlj = pair.KA_ModLJ(tree, sp["delta"])

        nvt = hoomd.md.methods.NVT(hoomd.filter.All(), float(temp), 0.1)
        integrator.forces = [mlj]
        integrator.methods = [nvt]

        sim.operations.integrator = integrator

        log = hoomd.logging.Logger()
        log.add(sim, quantities=["timestep"])

        sim.run(0)

        usteps = 100 # project.doc["usteps"]

        gsd_writer = hoomd.write.GSD(filename=job.fn(f'{out_dir}/traj.gsd'),
                                    trigger=hoomd.trigger.Periodic(usteps, phase=sim.timestep),
                                    mode='wb',
                                    log=log,
                                    )
        sim.operations.writers.append(gsd_writer)

        for i in range(10):
            print("step", i, end=" ", flush=True)
            sim.run(usteps * 1000)

        pathlib.Path(job.fn(done)).touch()
        print("\tDone!", temp)
        
        del mlj, tree, nvt, integrator, log, sim
        gc.collect()

Job {'N': 32768, 'replica': 0, 'delta': 0.0, 'phi': 1.2}
	Temp: 0.45
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.45
	Temp: 0.47
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.47
	Temp: 0.49
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.49
	Temp: 0.51
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.51
	Temp: 0.53
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.53
	Temp: 0.55
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.55
	Temp: 0.58
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.58
	Temp: 0.61
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.61
	Temp: 0.64
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.64
	Temp: 0.67
step 0 step 1 step 2 step 3 step 4 step 5 step 6 step 7 step 8 step 9 	Done! 0.67
